# Functions in Python

Reference notebook covering `def`/`return`, arguments, scope, recursion, `*args`/`**kwargs`, and closures.

## `def` and `return`

Functions package reusable behavior. `return` sends a value to the caller and exits the function.

In [1]:
def add(a, b):
    return a + b

print(add(2, 3))

5


## Positional and keyword arguments

In [2]:
def introduce(name, age, city):
    return f"{name}, {age}, {city}"

print(introduce("Alice", 21, "Hyderabad"))
print(introduce(city="Delhi", age=22, name="Bob"))

Alice, 21, Hyderabad
Bob, 22, Delhi


## Default arguments and mutable-default edge case

In [3]:
def greet(name, greeting="Hello"):
    return f"{greeting}, {name}!"

print(greet("Alice"))

Hello, Alice!


In [4]:
def bad_append(value, items=[]):
    items.append(value)
    return items

print(bad_append("a"))
print(bad_append("b"))  # Same list is reused

['a']
['a', 'b']


In [5]:
def safe_append(value, items=None):
    if items is None:
        items = []
    items.append(value)
    return items

print(safe_append("a"))
print(safe_append("b"))

['a']
['b']


## Positional-only and keyword-only parameters

In [6]:
def divide(a, b, /):
    return a / b

print(divide(10, 2))
# divide(a=10, b=2)  # TypeError

5.0


In [7]:
def configure(name, *, debug=False):
    return name, debug

print(configure("app", debug=True))
# configure("app", True)  # TypeError

('app', True)


## Scope and LEGB

Python resolves names using Local → Enclosing → Global → Built-in.

In [8]:
value = "global"

def outer():
    value = "enclosing"
    def inner():
        value = "local"
        print(value)
    inner()

outer()

local


In [9]:
value = 10

def change():
    value = 20
    print(value)

change()
print(value)

20
10


### `global` and `nonlocal`

In [10]:
counter = 0

def increment():
    global counter
    counter += 1

increment()
increment()
print(counter)

2


In [11]:
def make_counter(start=0):
    count = start
    def increment(step=1):
        nonlocal count
        count += step
        return count
    return increment

counter = make_counter(10)
print(counter())
print(counter())
print(counter(5))

11
12
17


## `*args` and `**kwargs`

In [12]:
def total(*args):
    print(type(args))
    return sum(args)

print(total(1, 2, 3))

<class 'tuple'>
6


In [13]:
def show_profile(**kwargs):
    print(type(kwargs))
    for key, value in kwargs.items():
        print(key, value)

show_profile(name="Alice", age=21, role="Engineer")

<class 'dict'>
name Alice
age 21
role Engineer


In [14]:
def flexible(name, *args, debug=False, **kwargs):
    print(name, args, debug, kwargs)

flexible("Alice", 1, 2, debug=True, role="AI")

Alice (1, 2) True {'role': 'AI'}


## Argument unpacking

In [15]:
def add(a, b, c):
    return a + b + c

values = [10, 20, 30]
print(add(*values))

60


In [16]:
def create_user(name, age, role):
    return {"name": name, "age": age, "role": role}

data = {"name": "Alice", "age": 21, "role": "Engineer"}
print(create_user(**data))

{'name': 'Alice', 'age': 21, 'role': 'Engineer'}


## Recursion

A recursive function calls itself and needs a base case.

In [17]:
def countdown(n):
    if n <= 0:
        print("Done")
        return
    print(n)
    countdown(n - 1)

countdown(5)

5
4
3
2
1
Done


In [18]:
def factorial(n):
    if n < 0:
        raise ValueError("factorial requires a non-negative integer")
    if n == 0:
        return 1
    return n * factorial(n - 1)

print(factorial(5))

120


In [19]:
import sys
print("Recursion limit:", sys.getrecursionlimit())
# def infinite():
#     infinite()
# infinite()  # RecursionError

Recursion limit: 3000


## Recursion with nested structures

In [20]:
def flatten(values):
    result = []
    for value in values:
        if isinstance(value, list):
            result.extend(flatten(value))
        else:
            result.append(value)
    return result

print(flatten([1, [2, 3], [4, [5, 6]]]))

[1, 2, 3, 4, 5, 6]


## Closures

A closure is a function that retains access to variables from its enclosing scope after the outer function returns.

In [21]:
def make_multiplier(factor):
    def multiply(number):
        return number * factor
    return multiply

double = make_multiplier(2)
triple = make_multiplier(3)
print(double(10))
print(triple(10))

20
30


In [22]:
def make_counter(start=0):
    count = start
    def increment():
        nonlocal count
        count += 1
        return count
    return increment

a = make_counter()
b = make_counter()
print(a(), a())
print(b())

1 2
1


## Inspecting closure state

In [23]:
def make_multiplier(factor):
    def multiply(number):
        return number * factor
    return multiply

double = make_multiplier(2)
print(double.__closure__)
print(double.__closure__[0].cell_contents)

(<cell at 0x000001D908D13460: int object at 0x00007FFFB4CD73C8>,)
2


## Closure late binding

In [24]:
functions = []
for i in range(3):
    functions.append(lambda: i)

print([f() for f in functions])

[2, 2, 2]


In [25]:
functions = []
for i in range(3):
    functions.append(lambda i=i: i)

print([f() for f in functions])

[0, 1, 2]


## Functions are objects and higher-order functions

In [26]:
def square(x):
    return x * x

operation = square
print(operation(5))
print(square is operation)

25
True


In [27]:
def apply_operation(value, operation):
    return operation(value)

print(apply_operation(5, square))

25


In [28]:
def choose_operation(name):
    if name == "add":
        def add(a, b): return a + b
        return add
    if name == "multiply":
        def multiply(a, b): return a * b
        return multiply
    raise ValueError("Unknown operation")

operation = choose_operation("multiply")
print(operation(4, 5))

20


## Type annotations and docstrings

In [29]:
def add(a: int, b: int) -> int:
    """Return the sum of two values."""
    return a + b

print(add(2, 3))
print(add.__doc__)

5
Return the sum of two values.


Annotations document intent; Python does not automatically enforce them at runtime.

## Function introspection

In [30]:
import inspect

def example(a, b=10, *, debug=False):
    return a + b

print(example.__name__)
print(example.__defaults__)
print(inspect.signature(example))

example
(10,)
(a, b=10, *, debug=False)


## Practical example: configurable scorer

In [31]:
def make_scorer(weight):
    def score(value):
        return value * weight
    return score

accuracy = make_scorer(0.7)
f1 = make_scorer(0.3)
print(accuracy(90))
print(f1(80))

62.99999999999999
24.0


## Practical example: flexible logging

In [32]:
def log(message, *tags, **metadata):
    print("MESSAGE:", message)
    if tags: print("TAGS:", tags)
    if metadata: print("METADATA:", metadata)

log("Model trained", "training", "experiment", epoch=10, accuracy=0.94)

MESSAGE: Model trained
TAGS: ('training', 'experiment')
METADATA: {'epoch': 10, 'accuracy': 0.94}


## Practical example: recursive tree traversal

In [33]:
tree = {"name":"root", "children":[{"name":"A","children":[]},{"name":"B","children":[{"name":"B1","children":[]},{"name":"B2","children":[]}]}]}

def print_tree(node, depth=0):
    print("  " * depth + node["name"])
    for child in node["children"]:
        print_tree(child, depth + 1)

print_tree(tree)

root
  A
  B
    B1
    B2


## Common pitfalls

- Missing `return` produces `None`.
- Mutable default arguments are shared between calls.
- `*args` is a tuple; `**kwargs` is a dictionary.
- Recursive functions need a terminating base case.
- Deep recursion can raise `RecursionError`.
- `global` modifies module-level state; `nonlocal` modifies enclosing state.
- Closures retain references to enclosing variables.
- Loop-created closures can show late binding.
- Type annotations do not enforce types at runtime.
- Argument order and function signatures matter.

## Quick reference

In [34]:
def add(a, b):
    return a + b

def greet(name, greeting="Hello"):
    return f"{greeting}, {name}"

def total(*args):
    return sum(args)

def configure(**kwargs):
    return kwargs

def flexible(name, *args, debug=False, **kwargs):
    pass

def positional_only(value, /):
    pass

def keyword_only(*, value):
    pass

def make_multiplier(factor):
    def multiply(x):
        return x * factor
    return multiply

def factorial(n):
    if n == 0:
        return 1
    return n * factorial(n - 1)

## Summary

Python functions combine reusable behavior, flexible argument handling, lexical scope, recursion, and closures. The key ideas are function signatures, `return`, positional/keyword arguments, LEGB scope, `*args`/`**kwargs`, recursive base cases, and retained enclosing state.